<a href="https://colab.research.google.com/github/azizbekreimbaev/cv/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
print("Menu detector")

Menu detector


In [45]:
from google.colab import drive
import torchvision.transforms as transforms
from torchvision import transforms

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.models import mobilenet_v2

import os
import numpy as np

In [32]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
DATASET_PATH = '/content/drive/MyDrive/food_dataset'


In [34]:
CUSTOM_CLASS_MAPPING = {
    "hot_dog": "hot_dog",
    "pizza": "pizza",
    "chocolate_cake": "dessert", #label grouping | class consolidation
    "cheesecake": "dessert", #label grouping | class consolidation
}

CLASSES = ['hot_dog', 'dessert', 'pizza']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)


print(CLASS_TO_IDX)

#TRANSFORM compose method(is like pipeline) helps to transform by order

transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])





{'hot_dog': 0, 'dessert': 1, 'pizza': 2}


Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

In [35]:

# =========================
# Custom Dataset Class
# =========================

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        print('image_path', img_path)

        label = self.labels[idx]
        print('label', label)

        try:
            image = Image.open(img_path).convert('RGB')

        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))

        if self.transform:
            image = self.transform(image)

        return image, label

In [36]:
# =========================
# Gather and Split Data
# =========================

all_images = []

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)  # /content/drive/MyDrive/food101_dataset
    print('class_path:', class_path)

    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue

    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)

split = int(0.8 * len(all_images))

train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)

print(len(dataset))

img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food_dataset/hot_dog
class_path: /content/drive/MyDrive/food_dataset/pizza
class_path: /content/drive/MyDrive/food_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food_dataset/cheesecake
images_length 3200
3200
image_path /content/drive/MyDrive/food_dataset/cheesecake/2559262.jpg
label 1


In [37]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)


In [38]:
print("transform:", "transform" in globals())
print("FoodDataset:", "FoodDataset" in globals())
print("train_images:", "train_images" in globals())
print("train_labels:", "train_labels" in globals())

transform: True
FoodDataset: True
train_images: True
train_labels: True


In [42]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)  # Parallel loading for speed

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)


images_length 3200
images_length 3200


In [46]:
# =========================
# Pretrained Model
# =========================

model = mobilenet_v2(weights="IMAGENET1K_V1")

# Replace final classification layer
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 113MB/s]


In [48]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)
model = model.to(device)

device cuda


In [49]:
criterion = nn.CrossEntropyLoss() #Loss function

optimizer = optim.Adam(model.parameters(), lr=0.001)
torch.backends.cudnn.benchmark = True